# Lab 11A: RNN versus LSTM

This week's lab consists of **two short notebooks** that follow the story of the lecture: each model is
motivated by a weakness of the previous one.

| Notebook | Question |
|---|---|
| **11A** RNN versus LSTM | Why can a vanilla RNN not remember what it saw 20 steps ago, and how does the LSTM fix that? |
| **11B** The seq2seq bottleneck | Why does a seq2seq model forget the beginning of long inputs, and how does attention fix that? |

Both notebooks use **toy tasks on random sequences of digits**, so that each model trains in seconds to
a few minutes and the effects from the lecture are easy to see. Use the remaining time for your project.

**Theory ↔ code.** As in Labs 9 and 10, boxes marked **🔗 Theory ↔ code** show how the equations from
the lecture map to lines of PyTorch.

**Cells marked 🛠️ TASK contain gaps (`???`) that you must fill in.** Each comes with a test cell; don't
move on until the tests pass. Write down your ✍️ **Prediction** before running an experiment.

---
## This notebook

| Part | You will… |
|---|---|
| 1. A vanilla RNN | implement the RNN update $\mathbf{h}_t = \tanh(\mathbf{W}\mathbf{h}_{t-1} + \mathbf{V}\mathbf{x}_t + \mathbf{b})$ and the loop over time |
| 2. How far back can it remember? | train an RNN and an LSTM on longer and longer sequences |
| 3. Why? | look at the gradient that reaches each timestep: the vanishing gradient problem |
| 4. Exam-style questions | check your understanding |

## Setup

**The task.** The input is a sequence of $T$ random digits from $\{0, \ldots, 7\}$, for example
`5 2 7 7 0 1 3 …`. The model reads the whole sequence and must then output the **first** digit (here `5`).
This is a **many-to-one** task, like sentiment analysis: one output after reading the whole sequence.

The task is trivial if the model can remember what it saw $T$ steps ago, and impossible otherwise:
then it can only guess, and is right 1 time in 8 (**12.5%: chance level**). By changing $T$ we can measure
how far back a model can remember.

The setup cell defines three helpers:

| Helper | Does |
|---|---|
| `make_batch(n, T)` | `n` random sequences of length `T` (as integer tokens), and their targets (the first token) |
| `train(model, T)` | trains `model` on sequences of length `T` (1,500 steps of Adam) and returns its accuracy on 2,000 new sequences |
| `gradient_per_timestep(model, T)` | the size of the gradient of the loss with respect to the input at each timestep $t$ (Part 3) |

In [ ]:
import math
import random
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


VOCAB = 8          # the digits 0..7
CHANCE = 1 / VOCAB


def make_batch(n, T):
    tokens = torch.randint(0, VOCAB, (n, T), device=device)   # (n, T) random digits
    return tokens, tokens[:, 0]                               # the target is the first digit


def train(model, T, steps=1500, lr=3e-3, seed=0):
    set_seed(seed)
    model.to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(steps):
        tokens, target = make_batch(128, T)
        loss = F.cross_entropy(model(tokens), target)
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    tokens, target = make_batch(2000, T)
    with torch.no_grad():
        accuracy = (model(tokens).argmax(1) == target).float().mean().item()
    return accuracy


def gradient_per_timestep(model, T, seed=0):
    # || dLoss / dx_t || for t = 1..T: how much the loss can "feel" the input at timestep t
    set_seed(seed)
    model.to(device)
    tokens, target = make_batch(256, T)
    x = model.embed(tokens)              # (256, T, embed_dim): the inputs x_1 .. x_T
    x.retain_grad()                      # ask PyTorch to keep the gradient with respect to x
    loss = F.cross_entropy(model.run(x), target)
    loss.backward()
    return x.grad.norm(dim=2).mean(0).cpu()   # (T,)

---
# Part 1 — A vanilla RNN

The RNN reads the sequence one token at a time and updates its **hidden state** $\mathbf{h}_t$, which
summarises everything seen so far (Lecture 11, Section 2.1):

$$\mathbf{h}_t = \tanh(\mathbf{W} \cdot \mathbf{h}_{t-1} + \mathbf{V} \cdot \mathbf{x}_t + \mathbf{b}), \qquad \mathbf{h}_0 = \mathbf{0}$$

For a many-to-one task, only the **last** hidden state is used for the prediction:

$$\hat{\mathbf{y}} = f_\text{MLP}(\mathbf{h}_T)$$

The input $\mathbf{x}_t$ is the **word embedding** of token $t$ (`nn.Embedding`, Lecture 11, Code 1).

### 🛠️ TASK 1: Implement the RNN update and the loop over time

> **🔗 Theory ↔ code**
>
> | Lecture | Code |
> |---|---|
> | $\mathbf{W} \cdot \mathbf{h}_{t-1}$ | `self.W(h)` (a linear layer without bias) |
> | $\mathbf{V} \cdot \mathbf{x}_t + \mathbf{b}$ | `self.V(x)` (a linear layer; its bias is $\mathbf{b}$) |
> | $\tanh(\cdot)$ | `torch.tanh(...)` |
> | $\mathbf{h}_t = f_\text{RNN}(\mathbf{h}_{t-1}, \mathbf{x}_t)$ | `h = self.cell(h, x[:, t])` (`x[:, t]` is $\mathbf{x}_t$ for the whole batch) |
> | $\hat{\mathbf{y}} = f_\text{MLP}(\mathbf{h}_T)$ | `self.out(h)` after the loop |

In [ ]:
class RNNCell(nn.Module):
    # One step of a vanilla RNN: h_t = tanh(W h_{t-1} + V x_t + b)
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.W = nn.Linear(hidden_dim, hidden_dim, bias=False)   # W
        self.V = nn.Linear(input_dim, hidden_dim)                # V and b

    def forward(self, h, x):
        return ???                                    # h_t = tanh(W h_{t-1} + V x_t + b)


class RNNClassifier(nn.Module):
    def __init__(self, vocab_size=VOCAB, embed_dim=16, hidden_dim=64):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embed = nn.Embedding(vocab_size, embed_dim)          # word embeddings
        self.cell = RNNCell(embed_dim, hidden_dim)                # f_RNN (the same at every timestep)
        self.out = nn.Linear(hidden_dim, vocab_size)              # f_MLP

    def run(self, x):
        # x: (batch, T, embed_dim) -- the embedded sequence x_1 .. x_T
        h = torch.zeros(x.shape[0], self.hidden_dim, device=x.device)   # h_0 = 0
        for t in range(x.shape[1]):
            h = ???                                   # h_t = f_RNN(h_{t-1}, x_t)
        return ???                                    # y_hat = f_MLP(h_T)

    def forward(self, tokens):
        return self.run(self.embed(tokens))

In [ ]:
# Test for Task 1: compare with PyTorch's own RNN, using the same weights
_m = RNNClassifier()
_ref = nn.RNN(16, 64, batch_first=True)
with torch.no_grad():
    _ref.weight_ih_l0.copy_(_m.cell.V.weight); _ref.bias_ih_l0.copy_(_m.cell.V.bias)
    _ref.weight_hh_l0.copy_(_m.cell.W.weight); _ref.bias_hh_l0.zero_()
    _x = torch.randn(3, 7, 16)
    _out = _m.run(_x)
    assert _out.shape == (3, VOCAB), f'run should return one prediction per sequence, shape (3, {VOCAB}); got {tuple(_out.shape)}'
    _h_seq, _ = _ref(_x)
    assert torch.allclose(_m.cell(torch.zeros(3, 64), _x[:, 0]), _h_seq[:, 0], atol=1e-5), 'RNNCell does not compute tanh(W h + V x + b)'
    assert torch.allclose(_out, _m.out(_h_seq[:, -1]), atol=1e-5), 'run should loop over all timesteps and use the LAST hidden state'
print('✅ Task 1 passed')

### The LSTM

For the LSTM we use PyTorch's `nn.LSTM`, which implements the LSTM of Lecture 11, Section 3 (you don't need the details of
Section 3.2).
It returns the hidden state at every timestep; for our many-to-one task we use only the last one, exactly
as for the RNN. (The two lines marked `forget gate` are explained in the optional Part 3; ignore them for now.)

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size=VOCAB, embed_dim=16, hidden_dim=64, forget_bias=2.0):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.out = nn.Linear(hidden_dim, vocab_size)
        with torch.no_grad():                                         # forget gate: see Part 3
            self.lstm.bias_ih_l0[hidden_dim:2 * hidden_dim].fill_(forget_bias)
            self.lstm.bias_hh_l0[hidden_dim:2 * hidden_dim].fill_(0.0)

    def run(self, x):
        hidden_states, _ = self.lstm(x)          # (batch, T, hidden_dim): h_1 .. h_T
        return self.out(hidden_states[:, -1])    # y_hat = f_MLP(h_T)

    def forward(self, tokens):
        return self.run(self.embed(tokens))


count = lambda m: sum(p.numel() for p in m.parameters())
print(f'RNN parameters:  {count(RNNClassifier()):,}')
print(f'LSTM parameters: {count(LSTMClassifier()):,}')

### Questions 1

1. Where in `RNNClassifier` do you see that the **same** parameters are used at every timestep? Which
   property of CNNs is this similar to?
2. Does the number of parameters change if we train on sequences of length 40 instead of 5? Why (not)?
3. *(Optional: requires Section 3.2 of the lecture notes.)* Why does the LSTM have about four times as many parameters in its
   recurrent layer as the RNN?
4. Why can the loop not be replaced by computing all the $\mathbf{h}_t$ in parallel?

---
# Part 2 — How far back can it remember?

We train an RNN and an LSTM for $T = 5, 10, 20, 40$. Each model sees the same number of training sequences.

✍️ **Prediction:** sketch the accuracy as a function of $T$ for both models. The lecture says that vanilla
RNNs "practically cannot learn dependencies spanning more than about 10–20 tokens".

In [ ]:
LENGTHS = [5, 10, 20, 40]
results = {'RNN': [], 'LSTM': []}
t0 = time.time()
for T in LENGTHS:
    results['RNN'].append(train(RNNClassifier(), T))
    results['LSTM'].append(train(LSTMClassifier(), T))
    print(f'T = {T:2d}:   RNN {results["RNN"][-1]:.0%}   LSTM {results["LSTM"][-1]:.0%}   ({time.time() - t0:.0f} s)')

plt.figure(figsize=(6, 4))
for name, acc in results.items():
    plt.plot(LENGTHS, acc, 'o-', label=name)
plt.axhline(CHANCE, color='grey', ls='--', label='chance (guessing)')
plt.xscale('log'); plt.xticks(LENGTHS, LENGTHS)
plt.xlabel('sequence length T'); plt.ylabel('accuracy'); plt.ylim(0, 1.05)
plt.title('Remember the first token'); plt.legend(); plt.show()

### Questions 2

1. Up to which length does the RNN solve the task? What does an accuracy of about 12.5% mean?
2. The RNN fails at $T = 40$ although the information it needs was in its input. Is the problem that the
   hidden state is too small to store one digit? (Hint: 64 numbers, one digit out of 8.)
3. How does the LSTM compare?

---
# Part 3 (optional) — Why? The gradient that reaches each timestep

> **This part is not mandatory.** It explains the results of Part 2 a little deeper than the exam
> requires. Do it if you have time.

**The RNN is a very deep network.** Unrolled over 40 timesteps, an RNN is a 40-layer network that uses the
same weights in every layer. To learn to keep the first token, the model needs a gradient that tells it how
the loss depends on the first input. **Backpropagation through time** passes this gradient backwards from
the last timestep to the first, one step at a time, exactly as backpropagation passes it backwards through
the layers of a deep network. At every step back, the gradient is multiplied by a factor that contains the
derivative of $\tanh$, which is at most 1. Many factors smaller than 1 multiplied together give a tiny
number: the gradient **vanishes**, just as in a deep network without skip connections (Lectures 3 and 5).

**The LSTM is like a ResNet.** A ResNet fixes this with skip connections, $\mathbf{x} + F(\mathbf{x})$: the
gradient can flow back through the "+" without shrinking. The LSTM has a skip connection **through time**:
its memory, the cell state, is updated by *adding* new information to what it remembers (the lecture makes the
same comparison between LSTMs and ResNets):

$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \text{(new information)}$$

Here $\mathbf{f}_t$, the **forget gate**, is a number between 0 and 1 that the LSTM learns. It decides how much
of its memory to keep at each step. If $\mathbf{f}_t \approx 1$, the memory, and the gradient, pass from one
step to the next almost unchanged, like through a ResNet skip connection. If $\mathbf{f}_t \approx 0.5$, half
is lost at every step.

**The experiment.** `gradient_per_timestep` measures how large the gradient is at every timestep $t$, for an
**untrained** model (as at the start of training). We compare the RNN with two LSTMs that differ only in where
their forget gate starts:

* `forget_bias=0.0`: the forget gate starts at about 0.5 (PyTorch's default);
* `forget_bias=2.0`: the forget gate starts at about 0.88 (what we used in Part 2; Keras does something similar
  by default).

✍️ **Prediction:** for which model does the gradient shrink fastest as we go back in time?

**Note:** the y-axis is logarithmic. A straight, falling line means that the gradient shrinks by the same
*factor* with every step back in time.

In [ ]:
T = 40
models = {
    'RNN': RNNClassifier(),
    'LSTM, forget gate starts at 0.5': LSTMClassifier(forget_bias=0.0),
    'LSTM, forget gate starts at 0.88': LSTMClassifier(forget_bias=2.0),
}
plt.figure(figsize=(7, 4))
for name, model in models.items():
    plt.semilogy(range(1, T + 1), gradient_per_timestep(model, T), label=name)
plt.xlabel('timestep t (1 = the first token, which the model must remember)')
plt.ylabel('size of the gradient')
plt.title('Gradient reaching each timestep (untrained models)'); plt.legend(); plt.show()

### 🔧 Knob

Train the LSTM whose forget gate starts at 0.5 on $T = 20$ and $T = 40$:

In [ ]:
for T in [20, 40]:
    print(f'T = {T}: LSTM with forget gate starting at 0.5: {train(LSTMClassifier(forget_bias=0.0), T):.0%}')

### Questions 3 (optional)

1. Read the plot for the RNN: roughly how much smaller is the gradient at $t = 1$ than at $t = 40$? Use
   this to explain your results from Part 2.
2. Why does the RNN's gradient shrink by roughly the same factor with every step back in time?
3. The LSTM whose forget gate starts at 0.5 also has a vanishing gradient, and fails to learn the task in
   the knob experiment. Explain why, and why a forget gate close to 1 fixes this.
4. What do the LSTM and the ResNet have in common?

---
# Part 4 — Exam-style questions

**Q1.** A vanilla RNN with hidden size 64 is trained on sequences of length 10. If we instead train it on
sequences of length 100, the number of parameters

* (a) grows by a factor 10.
* (b) stays the same.
* (c) grows by a factor 100.
* (d) depends on the vocabulary size only.

**Q2.** Why does the gradient of the loss with respect to the first input of a long sequence become very
small in a vanilla RNN?

* (a) Because the hidden state is reset to zero at every timestep.
* (b) Because backpropagation through time multiplies many factors, each containing the derivative of tanh, which is at most 1.
* (c) Because the learning rate is too small.
* (d) Because the softmax at the output saturates.

**Q3.** *(Not mandatory: requires Section 3.2 of the lecture notes.)* Which LSTM equation provides the "gradient highway" through time?

* (a) $\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$
* (b) $\mathbf{o}_t = \sigma(\mathbf{W}_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o)$
* (c) $\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t$
* (d) $\tilde{\mathbf{c}}_t = \tanh(\mathbf{W}_c [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c)$

**Q4.** In this notebook's task, a model reaches 12.5% accuracy on sequences of length 40. What does this tell you?

* (a) The model remembers the first token in 12.5% of the cases because its hidden state is too small.
* (b) The model performs at chance level: it has not learned to use the first token at all.
* (c) The model is overfitting.
* (d) The model needs more parameters.